# IN14: FinOps for GenAI and Governance

**Programme:** Advanced Agentic AI -- Production Engineering
**Track:** India | Walmart Global Tech Academy
**Module:** 5 -- AI Economics, Optimization and Architecture Review

## Objectives

By the end of this notebook you will be able to:

- Define a cost budget framework for a GenAI system in production
- Set governance thresholds that trigger automated alerts and spending caps
- Design auto-scaling policies that balance cost and latency
- Build a cost allocation model that attributes spend to teams and projects
- Implement a chargeback report for enterprise multi-team AI deployments
- Produce a FinOps governance dashboard specification

In [1]:
# import subprocess, sys
# subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
#                 'openai', 'python-dotenv'], check=True)
# print('Packages ready.')

In [2]:
import os, json, time, uuid
from pathlib import Path
from datetime import datetime, timezone, timedelta
from dotenv import load_dotenv
from openai import OpenAI
from collections import defaultdict

load_dotenv(override=True)
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
print('Client ready.')

Client ready.


## Section 1: FinOps for GenAI -- What It Is and Why It Matters

### What is FinOps?

FinOps (Financial Operations) is the practice of bringing financial accountability
to cloud and AI spending. For GenAI systems, FinOps means understanding, tracking,
and controlling token costs at the team, project, and feature level.

**Why GenAI needs dedicated FinOps:**

Traditional cloud costs (compute, storage) scale predictably with infrastructure.
GenAI costs scale with usage AND with prompt design decisions -- a 200-token
increase in the system prompt at 1 million daily calls is an invisible cost
that no infrastructure monitor will catch.

**The GenAI FinOps framework has four layers:**

| Layer | What it covers | Owner |
|---|---|---|
| Visibility | Track every token: by model, feature, user segment | ML Platform team |
| Budgets | Set daily/monthly spend limits per team and project | Engineering Manager |
| Governance thresholds | Automated alerts and caps before budget is exceeded | FinOps / Platform |
| Optimisation | Routing, caching, compression (covered in IN13) | ML Engineer |

**Cost allocation dimensions for Walmart Retail Assistant:**

| Dimension | Examples |
|---|---|
| Team | Search team, Supply Chain team, Customer Support team |
| Feature | Price lookup, Order tracking, Policy QA, Store hours |
| Environment | Dev, Staging, Production |
| Model | gpt-4-turbo, gpt-4o, gpt-4o-mini |

**What to remember:**
- Tag every LLM call with team, feature, and environment at the time of the call.
  Retrofitting tags onto historical logs is extremely difficult.
- Set budgets before launch, not after the first invoice.
- A governance threshold is a hard cap. An alert is a soft warning.
  You need both: alerts for investigation, caps to prevent runaway spend.

In [3]:
MODEL_PRICING = {
    'gpt-4-turbo':  {'input': 10.00, 'output': 30.00},
    'gpt-4o':       {'input':  5.00, 'output': 15.00},
    'gpt-4o-mini':  {'input':  0.15, 'output':  0.60},
}

def token_cost(model: str, input_tokens: int, output_tokens: int) -> float:
    p = MODEL_PRICING.get(model, {'input': 0, 'output': 0})
    return round((input_tokens / 1_000_000) * p['input'] +
                 (output_tokens / 1_000_000) * p['output'], 8)


class CostLedger:
    """In-memory ledger that tracks API call costs by team, feature, and model."""

    def __init__(self):
        self._records: list = []

    def record(self, team: str, feature: str, env: str,
               model: str, input_tokens: int, output_tokens: int):
        cost = token_cost(model, input_tokens, output_tokens)
        self._records.append({
            'ts':            datetime.now(timezone.utc).isoformat(),
            'call_id':       str(uuid.uuid4())[:8],
            'team':          team,
            'feature':       feature,
            'env':           env,
            'model':         model,
            'input_tokens':  input_tokens,
            'output_tokens': output_tokens,
            'cost_usd':      cost,
        })
        return cost

    def total_cost(self, filters: dict = None) -> float:
        records = self._records
        if filters:
            for k, v in filters.items():
                records = [r for r in records if r.get(k) == v]
        return round(sum(r['cost_usd'] for r in records), 6)

    def by_dimension(self, dim: str) -> dict:
        breakdown = defaultdict(float)
        for r in self._records:
            breakdown[r[dim]] += r['cost_usd']
        return {k: round(v, 6) for k, v in sorted(breakdown.items(), key=lambda x: -x[1])}

    def call_count(self, filters: dict = None) -> int:
        records = self._records
        if filters:
            for k, v in filters.items():
                records = [r for r in records if r.get(k) == v]
        return len(records)

    @property
    def records(self):
        return list(self._records)


# Simulate a day of Walmart Retail Assistant calls across teams and features
import random
random.seed(42)

ledger = CostLedger()

TEAMS    = ['search', 'supply_chain', 'customer_support']
FEATURES = {'search': ['price_lookup', 'product_search'],
            'supply_chain': ['inventory_check', 'reorder_alert'],
            'customer_support': ['order_tracking', 'policy_qa', 'store_hours']}
MODELS   = ['gpt-4o-mini', 'gpt-4o-mini', 'gpt-4o-mini', 'gpt-4o', 'gpt-4-turbo']  # weighted

for _ in range(500):  # simulate 500 calls
    team    = random.choice(TEAMS)
    feature = random.choice(FEATURES[team])
    model   = random.choice(MODELS)
    in_tok  = random.randint(300, 800)
    out_tok = random.randint(40,  150)
    ledger.record(team, feature, 'production', model, in_tok, out_tok)

print(f'Total simulated cost (500 calls): ${ledger.total_cost():.4f}')
print()
print('Cost by team:')
for team, cost in ledger.by_dimension('team').items():
    pct = cost / ledger.total_cost() * 100
    print(f'  {team:<20} ${cost:.4f}  ({pct:.1f}%)')
print()
print('Cost by model:')
for model, cost in ledger.by_dimension('model').items():
    pct = cost / ledger.total_cost() * 100
    print(f'  {model:<20} ${cost:.4f}  ({pct:.1f}%)')
print()
print('Cost by feature:')
for feat, cost in ledger.by_dimension('feature').items():
    print(f'  {feat:<25} ${cost:.4f}')

Total simulated cost (500 calls): $1.2702

Cost by team:
  customer_support     $0.4633  (36.5%)
  supply_chain         $0.4045  (31.8%)
  search               $0.4023  (31.7%)

Cost by model:
  gpt-4-turbo          $0.7699  (60.6%)
  gpt-4o               $0.4583  (36.1%)
  gpt-4o-mini          $0.0420  (3.3%)

Cost by feature:
  price_lookup              $0.2320
  inventory_check           $0.2057
  reorder_alert             $0.1989
  product_search            $0.1704
  policy_qa                 $0.1659
  store_hours               $0.1612
  order_tracking            $0.1362


## Section 2: Cost Budget Framework

### What is a Cost Budget in GenAI FinOps?

A cost budget is a pre-approved spending limit for a GenAI workload over a defined period.
Budgets are set per team, per environment, and per time window (daily, monthly).

**Budget hierarchy for Walmart Global Tech:**

```
Organisation budget (monthly cap)
  Team budget (monthly cap per team)
    Environment budget (dev << staging << prod)
      Feature budget (per use case)
```

**Alert and cap thresholds:**

| Threshold | Action |
|---|---|
| 70% of budget consumed | INFO alert to team lead |
| 85% of budget consumed | WARNING alert to engineering manager |
| 100% of budget consumed | HARD CAP: return fallback responses, page on-call |

**Daily budget sizing formula:**
```
daily_budget = (monthly_budget / 30) * safety_factor
safety_factor = 1.2  (allow 20% burst above average daily spend)
```

**What to remember:**
- Set daily caps, not just monthly. A runaway feature can exhaust a monthly
  budget in hours without a daily cap.
- Dev and staging environments must have much lower budgets than production.
  Engineers testing prompts in dev should not incur production-scale costs.
- Budget enforcement must be in the critical path of the API gateway.
  A budget check that runs asynchronously cannot block a call that is already made.

In [4]:
class BudgetGovernor:
    """Enforces spend budgets per team and triggers alerts."""

    def __init__(self, budgets: dict, alert_thresholds: tuple = (0.70, 0.85, 1.0)):
        # budgets: {'team_name': daily_usd_limit}
        self.budgets          = budgets
        self.alert_thresholds = alert_thresholds
        self._spend: dict     = defaultdict(float)
        self._alerts: list    = []
        self._blocked: set    = set()

    def record_spend(self, team: str, cost_usd: float) -> dict:
        """Record spend and return budget status."""
        if team in self._blocked:
            return {'allowed': False, 'reason': 'BUDGET_EXHAUSTED', 'team': team}

        self._spend[team] += cost_usd
        budget  = self.budgets.get(team, float('inf'))
        ratio   = self._spend[team] / budget if budget > 0 else 0

        status = 'OK'
        for threshold in sorted(self.alert_thresholds, reverse=True):
            if ratio >= threshold:
                if threshold >= 1.0:
                    self._blocked.add(team)
                    status = 'BLOCKED'
                    self._alerts.append({
                        'ts': datetime.now(timezone.utc).isoformat(),
                        'team': team, 'level': 'CRITICAL',
                        'msg': f'Budget exhausted: ${self._spend[team]:.4f} / ${budget:.2f}',
                    })
                elif threshold >= 0.85:
                    status = 'WARNING'
                    self._alerts.append({
                        'ts': datetime.now(timezone.utc).isoformat(),
                        'team': team, 'level': 'WARNING',
                        'msg': f'85% of budget consumed: ${self._spend[team]:.4f} / ${budget:.2f}',
                    })
                else:
                    status = 'INFO'
                break

        return {
            'allowed':    status != 'BLOCKED',
            'status':     status,
            'spent_usd':  round(self._spend[team], 4),
            'budget_usd': budget,
            'pct':        round(ratio * 100, 1),
            'team':       team,
        }

    def budget_report(self) -> list:
        rows = []
        for team, budget in self.budgets.items():
            spent = self._spend.get(team, 0.0)
            ratio = spent / budget if budget > 0 else 0
            rows.append({
                'team':       team,
                'budget_usd': budget,
                'spent_usd':  round(spent, 4),
                'pct':        round(ratio * 100, 1),
                'status':     'BLOCKED' if team in self._blocked else ('WARNING' if ratio >= 0.85 else ('INFO' if ratio >= 0.70 else 'OK')),
            })
        return rows


# Daily budgets per team (USD)
DAILY_BUDGETS = {
    'search':           50.00,
    'supply_chain':     30.00,
    'customer_support': 40.00,
}
governor = BudgetGovernor(DAILY_BUDGETS)

# Replay ledger records through the governor
blocked_calls = 0
for rec in ledger.records:
    status = governor.record_spend(rec['team'], rec['cost_usd'])
    if not status['allowed']:
        blocked_calls += 1

print('Budget Status Report:')
print(f'{"Team":<22} {"Budget":<10} {"Spent":<10} {"% Used":<8} Status')
print('-' * 60)
for row in governor.budget_report():
    print(f'{row["team"]:<22} ${row["budget_usd"]:<9.2f} ${row["spent_usd"]:<9.4f} {row["pct"]:<8} {row["status"]}')
print()
print(f'Blocked calls: {blocked_calls}')
print(f'Alerts fired : {len(governor._alerts)}')
for alert in governor._alerts:
    print(f'  [{alert["level"]}] {alert["team"]}: {alert["msg"]}')

Budget Status Report:
Team                   Budget     Spent      % Used   Status
------------------------------------------------------------
search                 $50.00     $0.4023    0.8      OK
supply_chain           $30.00     $0.4045    1.3      OK
customer_support       $40.00     $0.4633    1.2      OK

Blocked calls: 0
Alerts fired : 0


## Section 3: Auto-Scaling Policies for GenAI

### What is Auto-Scaling in the Context of GenAI?

In GenAI, auto-scaling refers to dynamically adjusting concurrency limits,
model tier selection, and batch sizes based on current load and cost signals.
Unlike traditional compute auto-scaling (add more VMs), GenAI auto-scaling
primarily adjusts the model and caching strategy under load.

**Three scaling triggers:**

| Trigger | Signal | Response |
|---|---|---|
| High load | Requests > 1000/min | Route 80% to gpt-4o-mini; increase cache TTL |
| Budget pressure | Daily spend > 70% | Downgrade non-critical features to cheaper model |
| SLO breach | P95 > 3000ms | Reduce K in retrieval; strip non-essential context |

**Scaling rules for Walmart Retail Assistant:**

```
NORMAL load (< 500 req/min):
  model_mix = 70% mini, 20% 4o, 10% turbo
  cache_ttl = 3600s (1 hour)
  retrieval_k = 3

HIGH load (500 - 2000 req/min):
  model_mix = 90% mini, 10% 4o, 0% turbo
  cache_ttl = 7200s (2 hours, more aggressive)
  retrieval_k = 2  (reduce context)

CRITICAL load (> 2000 req/min OR budget > 85%):
  model_mix = 100% mini
  serve from cache only for top-20 known queries
  queue non-cached complex queries
```

**What to remember:**
- Define load thresholds BEFORE launch based on capacity planning.
  Reactive scaling decisions made during an incident are error-prone.
- Log every scaling decision with the trigger signal and the action taken.
- Have a manual override. Auto-scaling must be overridable by the on-call engineer.

In [5]:
class AutoScalingPolicy:
    """Determines the operating mode based on load and budget signals."""

    MODES = {
        'NORMAL':   {'model_mini': 0.70, 'model_4o': 0.20, 'model_turbo': 0.10, 'cache_ttl': 3600,  'k': 3},
        'HIGH':     {'model_mini': 0.90, 'model_4o': 0.10, 'model_turbo': 0.00, 'cache_ttl': 7200,  'k': 2},
        'CRITICAL': {'model_mini': 1.00, 'model_4o': 0.00, 'model_turbo': 0.00, 'cache_ttl': 14400, 'k': 1},
    }

    def evaluate(self, requests_per_min: int, budget_pct: float) -> dict:
        if requests_per_min > 2000 or budget_pct >= 85:
            mode = 'CRITICAL'
        elif requests_per_min > 500 or budget_pct >= 70:
            mode = 'HIGH'
        else:
            mode = 'NORMAL'

        return {
            'mode':         mode,
            'config':       self.MODES[mode],
            'trigger':      f'rpm={requests_per_min}, budget_pct={budget_pct}%',
        }

policy = AutoScalingPolicy()
scenarios = [
    (300,  45, 'Tuesday mid-morning'),
    (800,  60, 'Wednesday lunchtime spike'),
    (1200, 71, 'Friday evening, 70% budget hit'),
    (2500, 88, 'Black Friday peak, budget pressure'),
]

print('Auto-Scaling Policy Decisions:')
print()
for rpm, pct, label in scenarios:
    result = policy.evaluate(rpm, pct)
    cfg    = result['config']
    print(f'Scenario: {label}')
    print(f'  Trigger  : {result["trigger"]}')
    print(f'  Mode     : {result["mode"]}')
    print(f'  Model mix: {cfg["model_mini"]*100:.0f}% mini / {cfg["model_4o"]*100:.0f}% 4o / {cfg["model_turbo"]*100:.0f}% turbo')
    print(f'  Cache TTL: {cfg["cache_ttl"]}s  |  Retrieval K: {cfg["k"]}')
    print()

Auto-Scaling Policy Decisions:

Scenario: Tuesday mid-morning
  Trigger  : rpm=300, budget_pct=45%
  Mode     : NORMAL
  Model mix: 70% mini / 20% 4o / 10% turbo
  Cache TTL: 3600s  |  Retrieval K: 3

Scenario: Wednesday lunchtime spike
  Trigger  : rpm=800, budget_pct=60%
  Mode     : HIGH
  Model mix: 90% mini / 10% 4o / 0% turbo
  Cache TTL: 7200s  |  Retrieval K: 2

Scenario: Friday evening, 70% budget hit
  Trigger  : rpm=1200, budget_pct=71%
  Mode     : HIGH
  Model mix: 90% mini / 10% 4o / 0% turbo
  Cache TTL: 7200s  |  Retrieval K: 2

Scenario: Black Friday peak, budget pressure
  Trigger  : rpm=2500, budget_pct=88%
  Mode     : CRITICAL
  Model mix: 100% mini / 0% 4o / 0% turbo
  Cache TTL: 14400s  |  Retrieval K: 1



## Section 4: Chargeback Models for Enterprise Teams

### What is a Chargeback Model?

A chargeback model attributes shared infrastructure costs back to the teams
or projects that consumed them. In enterprise AI, this means each team
receives a monthly bill for the token costs their features generated.

**Why chargeback matters:**
- Without chargeback, AI costs are invisible to the teams generating them.
  Engineers have no incentive to optimise prompts or use cheaper models.
- With chargeback, teams become cost-aware: a 10x prompt cost increase
  shows up directly on their team's budget.

**Three chargeback models:**

| Model | How it works | Best for |
|---|---|---|
| Direct attribution | Each team pays exactly what they consumed | Teams with clearly separated workloads |
| Proportional allocation | Shared platform cost split by usage % | Shared services (embedding, retrieval) |
| Flat rate | Fixed price per 1,000 API calls | Teams that want cost predictability |

**What to remember:**
- Shared services (Pinecone vector DB, embedding generation) must be
  split proportionally, not charged to one team.
- Always provide a cost forecast alongside the chargeback invoice.
  Teams cannot manage costs they cannot predict.
- Chargeback reports must be delivered monthly, not quarterly.
  Quarterly reports are too late to influence engineering decisions.

In [6]:
def chargeback_report(
    ledger: 'CostLedger',
    shared_platform_cost: float,
    period: str = 'July 2026',
) -> dict:
    """Generate a monthly chargeback report with direct and proportional attribution."""
    # Direct attribution: token costs per team
    direct = ledger.by_dimension('team')
    total_direct = sum(direct.values())

    # Proportional allocation of shared platform cost (Pinecone, embedding API)
    allocation = {}
    for team, cost in direct.items():
        share = cost / total_direct if total_direct > 0 else 0
        allocation[team] = round(share * shared_platform_cost, 4)

    # Final chargeback per team
    chargeback = {}
    for team in direct:
        chargeback[team] = {
            'direct_token_cost':     round(direct.get(team, 0), 4),
            'platform_allocation':   allocation.get(team, 0),
            'total_chargeback':      round(direct.get(team, 0) + allocation.get(team, 0), 4),
            'call_count':            ledger.call_count({'team': team}),
            'cost_per_call_usd':     round(direct.get(team, 0) / max(ledger.call_count({'team': team}), 1), 6),
        }

    return {
        'period':               period,
        'total_direct_usd':     round(total_direct, 4),
        'shared_platform_usd':  shared_platform_cost,
        'total_billed_usd':     round(total_direct + shared_platform_cost, 4),
        'by_team':              chargeback,
    }

report = chargeback_report(
    ledger=ledger,
    shared_platform_cost=12.50,  # simulated Pinecone + embedding costs
    period='July 2026',
)

print(f'CHARGEBACK REPORT -- {report["period"]}')
print('=' * 65)
print(f'Total direct token cost  : ${report["total_direct_usd"]:.4f}')
print(f'Shared platform cost     : ${report["shared_platform_usd"]:.2f} (Pinecone + Embedding)')
print(f'Total billed             : ${report["total_billed_usd"]:.4f}')
print()
print(f'{"Team":<22} {"Direct ($)":<12} {"Platform ($)":<14} {"Total ($)":<12} Calls  Cost/Call')
print('-' * 80)
for team, d in report['by_team'].items():
    print(f'{team:<22} {d["direct_token_cost"]:<12.4f} {d["platform_allocation"]:<14.4f} {d["total_chargeback"]:<12.4f} {d["call_count"]:<6} ${d["cost_per_call_usd"]:.6f}')

Path('chargeback_report_july_2026.json').write_text(json.dumps(report, indent=2))
print()
print('Saved: chargeback_report_july_2026.json')

CHARGEBACK REPORT -- July 2026
Total direct token cost  : $1.2702
Shared platform cost     : $12.50 (Pinecone + Embedding)
Total billed             : $13.7702

Team                   Direct ($)   Platform ($)   Total ($)    Calls  Cost/Call
--------------------------------------------------------------------------------
customer_support       0.4633       4.5596         5.0229       176    $0.002632
supply_chain           0.4045       3.9810         4.3855       166    $0.002437
search                 0.4023       3.9594         4.3617       158    $0.002546

Saved: chargeback_report_july_2026.json


## Section 5: Governance Thresholds and Policy Enforcement

### What are Governance Thresholds?

Governance thresholds are pre-defined limits that, when breached, trigger
automated enforcement actions -- not just alerts. They are the operational
enforcement layer of the FinOps framework.

**Governance thresholds for Walmart Retail Assistant (production):**

| Threshold | Limit | Action |
|---|---|---|
| Daily spend per team | $50 | Block requests, serve fallback |
| Monthly spend per team | $1,200 | Restrict to gpt-4o-mini only |
| Cost per call | > $0.05 | Log for review; flag for prompt audit |
| P95 latency | > 3000ms | Downgrade model, reduce K |
| Error rate | > 2% | Page on-call, enable circuit breaker |
| Toxicity score | > 0.30 | Block response, escalate to trust & safety |

**Governance policy document (what to produce for architecture review):**
- Who can change governance thresholds (requires VP-level approval)
- What the escalation path is when a hard cap is hit
- How exceptions are requested and approved
- SLA for resolving a budget block (target: 2 hours)

In [7]:
class GovernancePolicy:
    """Enforces cost and quality governance thresholds for production."""

    THRESHOLDS = {
        'cost_per_call_flag':    0.05,   # USD: flag calls above this for audit
        'monthly_restrict_usd':  1200,   # USD: restrict to mini-only above this
        'toxicity_block':        0.30,   # toxicity score: block above this
        'p95_latency_ms':        3000,   # ms: SLO breach
        'error_rate_pct':        2.0,    # %: page on-call
    }

    def __init__(self):
        self.violations: list = []
        self._monthly_spend: dict = defaultdict(float)
        self._restricted_teams: set = set()

    def check_call(self, team: str, cost_usd: float,
                   toxicity: float = 0.0, latency_ms: float = 0.0) -> dict:
        issues = []

        # Monthly spend restriction
        self._monthly_spend[team] += cost_usd
        if self._monthly_spend[team] > self.THRESHOLDS['monthly_restrict_usd']:
            self._restricted_teams.add(team)
            issues.append('MONTHLY_RESTRICT: downgrade to gpt-4o-mini')

        # Per-call cost flag
        if cost_usd > self.THRESHOLDS['cost_per_call_flag']:
            issues.append(f'HIGH_COST_CALL: ${cost_usd:.4f} exceeds flag threshold')

        # Toxicity block
        if toxicity > self.THRESHOLDS['toxicity_block']:
            issues.append(f'TOXICITY_BLOCK: score {toxicity} exceeds threshold')

        # Latency SLO
        if latency_ms > self.THRESHOLDS['p95_latency_ms']:
            issues.append(f'SLO_BREACH: {latency_ms}ms exceeds P95 target')

        if issues:
            self.violations.append({'team': team, 'issues': issues, 'cost_usd': cost_usd})

        return {
            'team':       team,
            'restricted': team in self._restricted_teams,
            'issues':     issues,
            'pass':       len(issues) == 0,
        }


gov = GovernancePolicy()

# Simulate governance checks
test_calls = [
    ('customer_support', 0.002, 0.01, 1200, 'Normal call'),
    ('customer_support', 0.08,  0.01, 1500, 'Expensive call (long prompt)'),
    ('search',           0.003, 0.35, 900,  'Toxic response flagged'),
    ('search',           0.003, 0.01, 3500, 'SLO breach'),
    ('supply_chain',     0.001, 0.01, 800,  'Normal call'),
]

print('Governance Policy Checks:')
for team, cost, tox, lat, label in test_calls:
    result = gov.check_call(team, cost, tox, lat)
    status = 'PASS' if result['pass'] else 'FAIL'
    print(f'  [{status}] {label}')
    if result['issues']:
        for issue in result['issues']:
            print(f'       {issue}')
print()
print(f'Total violations: {len(gov.violations)}')

Governance Policy Checks:
  [PASS] Normal call
  [FAIL] Expensive call (long prompt)
       HIGH_COST_CALL: $0.0800 exceeds flag threshold
  [FAIL] Toxic response flagged
       TOXICITY_BLOCK: score 0.35 exceeds threshold
  [FAIL] SLO breach
       SLO_BREACH: 3500ms exceeds P95 target
  [PASS] Normal call

Total violations: 3


## Summary

| Concept | Key Rule |
|---|---|
| Cost tagging | Tag every call with team, feature, environment at call time |
| Budget hierarchy | Org -> Team -> Environment -> Feature; daily caps are mandatory |
| Alert thresholds | 70% INFO | 85% WARNING | 100% HARD CAP |
| Auto-scaling | Downgrade model and increase cache TTL under load or budget pressure |
| Chargeback | Direct attribution for token costs; proportional for shared platform costs |
| Governance | Hard caps enforced in the critical path, not asynchronously |

**Next: IN15** -- Architecture Review Techniques: how to prepare for the ARB,
what questions to expect, and how to write a trade-off note that lasts.